# 🎬 Video Dubbing Pipeline for Google Colab

**Note** - This only works on runtime version 2025.10 (Colab)

**Dubbed Video Link** -  https://drive.google.com/drive/folders/1KA7jXcIEg9Zjl2UzGOSXXxx7aB1RG87i?usp=sharing

This notebook combines all the necessary code to run the video dubbing pipeline.

**Instructions:**
1.  **Set up the API Key:** In the "⚙️ 1. Configuration" step, enter your Google Gemini API key.
2.  **Upload Video:** Run the same cell to upload your video file (`.mp4`).
3.  **Run the Pipeline:** Execute the "🚀 2. Run the Dubbing Pipeline" cell. This will take a significant amount of time, especially the first time as it downloads the models.
4.  **View the Result:** Once the pipeline is complete, the final dubbed video will be displayed at the end.

## 📦 Step 0: Install Dependencies

In [ ]:
print("Installing dependencies... This may take a few minutes.")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q coqui-tts
!pip install -q demucs
!pip install -q moviepy
!pip install -q pydub
!pip install -q librosa
!pip install -q google-genai
!pip install -q python-decouple
!pip install git+https://github.com/openai/whisper.git
!pip install -q speechbrain
!pip install -q pyAudioAnalysis
!pip install -q indic-transliteration

print("✅ All dependencies installed.")

Installing dependencies... This may take a few minutes.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 112.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 17.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 63.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 31.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 33.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 124.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing

## 📚 Step 1: Import Libraries and Define Functions

In [ ]:
import os
import subprocess
import json
import re
import datetime
from typing import Optional
import sys
import tempfile

import torch
import whisper
import numpy as np
import librosa
import torchaudio
import soundfile as sf
from pydub import AudioSegment
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip
from TTS.api import TTS
import google.generativeai as genai
from decouple import config, UndefinedValueError
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from speechbrain.pretrained import SpeakerRecognition
from pyAudioAnalysis import ShortTermFeatures
from google.colab import files
from IPython.display import display, Video, HTML

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if 

In [ ]:
# Fix for deprecated np.complex
np.complex = complex

print("Loading AI Models... This may take some time on first run.")
# Load Whisper model
whisper_model = whisper.load_model("small")

# Load SpeechBrain speaker recognition model
speaker_model = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec"
)

# Load Coqui TTS model
device = "cuda" if torch.cuda.is_available() else "cpu"
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Models loaded.")

# --- Bangla Utilities ---
bangla_number = ["০", "১", "২", "৩", "৪", "৫", "৬", "৭", "৮", "৯"]
english_number = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
def convert_english_digit_to_bangla_digit(original):
    converted = ""
    for character in str(original):
        if character in english_number:
            converted += bangla_number[english_number.index(character)]
        else:
            converted += character
    return converted

# --- Audio Extraction ---
def extract_audio_from_video(video_path, output_audio_path):
    video_clip = VideoFileClip(video_path)
    audio_clip = video_clip.audio
    audio_clip.write_audiofile(output_audio_path)
    video_clip.close()
    return output_audio_path

def separate_vocals_with_demucs(audio_path):
    output_dir="extracted_audio"
    subprocess.run([
        "demucs", "--two-stems=vocals", audio_path, "-o", output_dir
    ], check=True, capture_output=True, text=True)
    # Demucs creates a subdirectory with the name of the model, e.g., 'htdemucs'
    model_name = 'htdemucs' # default model name
    base_name = os.path.splitext(os.path.basename(audio_path))[0]
    vocals_path = os.path.join(output_dir, model_name, base_name, "vocals.wav")
    music_path = os.path.join(output_dir, model_name, base_name, "no_vocals.wav")
    return vocals_path, music_path

def extraction_audio(video_input):
    print("1. Extracting audio from video...")
    audio_extracted = "extracted_audio/extracted_audio.wav"
    os.makedirs("extracted_audio", exist_ok=True)
    extract_audio_from_video(video_input, audio_extracted)

    print("2. Separating vocals and background music with Demucs...")
    vocals_path, music_path = separate_vocals_with_demucs(audio_extracted)

    print(f"- Vocals saved to: {vocals_path}")
    print(f"- Music saved to: {music_path}")
    return vocals_path, music_path

# --- Text Extraction & Analysis ---
def extract_audio_features(segment_wav):
    try:
        y, sr = librosa.load(segment_wav, sr=None)
        if len(y) < sr * 0.05: return {"avg_pitch": 0.0, "avg_energy": 0.0, "avg_loudness": 0.0}
        st_features, _ = ShortTermFeatures.feature_extraction(y, sr, 0.050 * sr, 0.025 * sr)
        if st_features.shape[1] == 0: return {"avg_pitch": 0.0, "avg_energy": 0.0, "avg_loudness": 0.0}
        energy = float(np.mean(st_features[1]))
        loudness = float(np.mean(librosa.feature.rms(y=y)))
        try:
            pitch = librosa.yin(y, fmin=50, fmax=300, sr=sr)
            avg_pitch = float(np.mean(pitch))
        except Exception: avg_pitch = 0.0
        return {"avg_pitch": avg_pitch, "avg_energy": energy, "avg_loudness": loudness}
    except Exception: return {"avg_pitch": 0.0, "avg_energy": 0.0, "avg_loudness": 0.0}

def transcribe_audio(audio_path):
    result = whisper_model.transcribe(audio_path)
    output_segments = []
    for seg in result["segments"]:
        text = seg["text"].strip()
        if any("\u0900" <= ch <= "\u097F" for ch in text):
            try: text_hinglish = transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
            except: text_hinglish = text
        else: text_hinglish = text
        output_segments.append({"start": seg["start"], "end": seg["end"], "text": text_hinglish})
    return output_segments

def classify_gender(segment_path):
    try:
        signal, fs = torchaudio.load(segment_path)
        embeddings = speaker_model.encode_batch(signal)
        vector = embeddings.squeeze().detach().cpu().numpy()
        score = np.mean(vector)
        if score > 0.01: return "MALE"
        elif score < -0.01: return "FEMALE"
        else: return "OTHER"
    except Exception: return "UNKNOWN"

def analyze_audio(audio_path):
    print("3. Transcribing audio and analyzing segments (gender, pitch, etc.)...")
    audio = AudioSegment.from_file(audio_path)
    segments = transcribe_audio(audio_path)
    if not isinstance(segments, list): return []
    output = []
    for seg in segments:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_audio:
            audio[int(seg['start'] * 1000):int(seg['end'] * 1000)].export(temp_audio.name, format="wav")
            temp_audio_path = temp_audio.name
        try:
            gender = classify_gender(temp_audio_path)
            features = extract_audio_features(temp_audio_path)
        finally:
            try: os.remove(temp_audio_path)
            except PermissionError: pass
        duration = seg['end'] - seg['start']
        speech_rate = len(seg['text'].split()) / max(duration, 1.0)
        output.append({
            "start_time": round(seg['start'], 2), "end_time": round(seg['end'], 2), "duration": round(duration, 2),
            "text": seg['text'].strip(), "speech_rate": round(speech_rate, 2), "gender": gender,
            "avg_pitch": round(features['avg_pitch'], 2), "avg_energy": round(features['avg_energy'], 2),
            "avg_loudness": round(features['avg_loudness'], 2)
        })
    print("- Transcription complete.")
    return output

# --- Audio Clipping ---
def clip_audio(transcription, audio_path):
    print("4. Clipping original audio for voice cloning references...")
    output_dir = "cliped_audio"
    os.makedirs(output_dir, exist_ok=True)
    audio = AudioSegment.from_file(audio_path)
    for segment in transcription:
        start_ms = int(segment["start_time"] * 1000)
        end_ms = int(segment["end_time"] * 1000)
        clip = audio[start_ms:end_ms]
        filename = f"{output_dir}/{segment['start_time']}_{segment['end_time']}.mp3"
        clip.export(filename, format="mp3")
    print("- Audio clips saved.")

# --- Translation ---
def translation(text, language, api_key):
    print(f"5. Translating text to {language} using Gemini API...")
    genai.configure(api_key=api_key)
    client = genai.GenerativeModel('gemini-2.5-flash')
    prompt = f'Translate the "text" in the following JSON array into {language}. Respond with ONLY the translated JSON array, keeping all other fields identical. Ensure numbers are written as words. Input: {json.dumps(text)}'
    response = client.generate_content(prompt)
    raw = response.text.strip()
    cleaned_json = re.sub(r"^```[a-zA-Z]*|```$", "", raw).strip()
    try:
        translated_data = json.loads(cleaned_json)
        print("- Translation successful.")
        return translated_data
    except json.JSONDecodeError as e:
        print(f"\n❌ JSON parsing failed: {e}\nRaw output:\n{cleaned_json}")
        return []

# --- Text to Speech ---
def match_duration(generated_file, target_duration):
    y, sr = librosa.load(generated_file, sr=None)
    current_duration = librosa.get_duration(y=y, sr=sr)
    if abs(current_duration - target_duration) < 0.05: return AudioSegment.from_file(generated_file)
    rate = current_duration / target_duration
    y_stretched = librosa.effects.time_stretch(y=y, rate=rate)
    temp_file = generated_file.replace(".wav", "_stretched.wav")
    sf.write(temp_file, y_stretched, sr)
    return AudioSegment.from_file(temp_file)

def text_to_speech_with_timestamps(corrected_transcription, audio_path, language):
    print("6. Generating new audio with cloned voice...")
    prevEnd = 0.0
    final_audio = AudioSegment.silent(duration=0)
    gen_audio_dir = os.path.join(audio_path, "generated_audio")
    os.makedirs(gen_audio_dir, exist_ok=True)
    try:
        for i, segment in enumerate(corrected_transcription):
            print(f"- Processing segment {i+1}/{len(corrected_transcription)}...")
            start, end, text = segment["start_time"], segment["end_time"], segment["text"]
            ref_audio_file = os.path.join(audio_path, f"{start}_{end}.mp3")
            if not os.path.exists(ref_audio_file): continue
            gen_file = os.path.join(gen_audio_dir, f"gen_{start}_{end}.wav")
            tts_model.tts_to_file(text=text, speaker_wav=ref_audio_file, language=language, file_path=gen_file)
            target_duration = end - start
            stretched_audio = match_duration(gen_file, target_duration)
            if prevEnd < start:
                silence_duration = (start - prevEnd) * 1000
                if silence_duration > 0: final_audio += AudioSegment.silent(duration=silence_duration)
            final_audio += stretched_audio
            prevEnd = end
        print("- New audio generated.")
        os.makedirs("final_audio", exist_ok=True)
        final_audio_path = "final_audio/finalAudio.wav"
        final_audio.export(final_audio_path, format="wav")
        return final_audio_path
    except Exception as e:
        print(f"❌ Error during TTS: {e}")
        return None

Loading AI Models... This may take some time on first run.


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 216MiB/s]
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

DEBUG:speechbrain.utils.fetching:Fetch: Local file found, creating symlink '/root/.cache/huggingface/hub/models--speechbrain--spkrec-ecapa-voxceleb/snapshots/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/hyperparams.yaml' -> '/content/pretrained_models/spkrec/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _load
DEBUG:speechbrain.utils.checkpoints:Registered parameter transfer hook for _load
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)

DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load_if_possible
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in pretrained_models/spkrec.
INF

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

...

label_encoder.txt: 0.00B [00:00, ?B/s]

DEBUG:speechbrain.utils.fetching:Fetch: Local file found, creating symlink '/root/.cache/huggingface/hub/models--speechbrain--spkrec-ecapa-voxceleb/snapshots/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/label_encoder.txt' -> '/content/pretrained_models/spkrec/label_encoder.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["label_encoder"] = /content/pretrained_models/spkrec/label_encoder.ckpt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): embedding_model -> /content/pretrained_models/spkrec/embedding_model.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): mean_var_norm_emb -> /content/pretrained_models/spkrec/mean_var_norm_emb.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): classifier -> /content/pretrained_models/spkrec/

 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y


100%|██████████| 1.87G/1.87G [00:21<00:00, 87.2MiB/s]
4.37kiB [00:00, 4.60MiB/s]
361kiB [00:00, 114MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 49.2kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 47.5MiB/s]


✅ Models loaded.


## ⚙️ Step 2: Configuration

1.  Add your Google Gemini API key in the `GEMINI_API_KEY` field.
2.  Specify the target language for the dubbing (e.g., `Hindi`, `Spanish`, `French`).
3.  Run the cell and click "Choose Files" to upload your video.

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
#@title Enter Configuration Details
if not GEMINI_API_KEY:
  GEMINI_API_KEY = ""  #@param {type:"string"}

if not GEMINI_API_KEY:
  raise ValueError("Please provide your Gemini API Key.")

print("Please upload a video file (.mp4)")
uploaded = files.upload()

if not uploaded:
  raise ValueError("No file uploaded. Please run the cell again and upload a file.")

input_video_path = list(uploaded.keys())[0]
print(f'\n✅ Uploaded "{input_video_path}" successfully.')

Please upload a video file (.mp4)


Saving rocky.mp4 to rocky.mp4

✅ Uploaded "rocky.mp4" successfully.


In [ ]:
TARGET_LANGUAGE = "English"  #@param {type:"string"}

## 🚀 Step 3: Run the Dubbing Pipeline

This cell executes the entire process. It will print updates for each major step.

In [ ]:
try:
    # Step 1 & 2: Extract Audio and Separate Vocals
    vocals_path, music_path = extraction_audio(input_video_path)

    # Step 3: Transcribe and Analyze
    transcription = analyze_audio(vocals_path)

    # Step 4: Clip Original Audio
    clip_audio(transcription, vocals_path)

    # Step 5: Translate Text
    corrected_transcription = translation(transcription, TARGET_LANGUAGE, GEMINI_API_KEY)

    # Step 6: Generate New Speech
    if corrected_transcription:
        language_code = TARGET_LANGUAGE.lower()[:2] # Use first two letters for TTS model (e.g., 'hi', 'es')
        dubbed_audio_path = text_to_speech_with_timestamps(corrected_transcription, 'cliped_audio', language_code)

        if dubbed_audio_path:
            # Step 7: Combine final audio (dubbed voice + background music) with video
            print("7. Combining dubbed audio, background music, and video...")
            video_clip = VideoFileClip(input_video_path)
            dubbed_audio_clip = AudioFileClip(dubbed_audio_path)
            music_audio_clip = AudioFileClip(music_path)

            # Lower music volume slightly to make voice clearer
            music_audio_clip = music_audio_clip.volumex(0.6)

            combined_audio = CompositeAudioClip([dubbed_audio_clip, music_audio_clip])
            final_video = video_clip.set_audio(combined_audio)

            final_video_path = "final_dubbed_video.mp4"
            final_video.write_videofile(final_video_path, codec="libx264", audio_codec="aac")
            print(f"🎬✅ Final video saved to {final_video_path}")

            # Clean up
            video_clip.close()
            dubbed_audio_clip.close()
            music_audio_clip.close()
        else:
            print("❌ Dubbed audio generation failed.")
    else:
        print("❌ Translation failed. Cannot proceed.")

except Exception as e:
    print(f"An error occurred during the pipeline: {e}")


1. Extracting audio from video...
MoviePy - Writing audio in extracted_audio/extracted_audio.wav


MoviePy - Done.
2. Separating vocals and background music with Demucs...
- Vocals saved to: extracted_audio/htdemucs/extracted_audio/vocals.wav
- Music saved to: extracted_audio/htdemucs/extracted_audio/no_vocals.wav
3. Transcribing audio and analyzing segments (gender, pitch, etc.)...


  warnings.warn(

  s = torchaudio.io.StreamReader(src, format, None, buffer_size)

  warnings.warn(

  s = torchaudio.io.StreamReader(src, format, None, buffer_size)



- Transcription complete.
4. Clipping original audio for voice cloning references...
- Audio clips saved.
5. Translating text to English using Gemini API...


  warnings.warn(

  s = torchaudio.io.StreamReader(src, format, None, buffer_size)



...

  warnings.warn(

  s = torchaudio.io.StreamReader(src, format, None, buffer_size)



- Processing segment 2/16...
- Processing segment 3/16...
- Processing segment 4/16...
- Processing segment 5/16...
- Processing segment 6/16...
- Processing segment 7/16...
- Processing segment 8/16...
- Processing segment 9/16...
- Processing segment 10/16...
- Processing segment 11/16...
- Processing segment 12/16...
- Processing segment 13/16...
- Processing segment 14/16...
- Processing segment 15/16...
- Processing segment 16/16...
- New audio generated.
7. Combining dubbed audio, background music, and video...
Moviepy - Building video final_dubbed_video.mp4.
MoviePy - Writing audio in final_dubbed_videoTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video final_dubbed_video.mp4



t:  99%|█████████▉| 1068/1074 [00:15<00:00, 88.06it/s, now=None]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:123: UserWarning: Warning: in file rocky.mp4, 691200 bytes wanted but 0 bytes read,at frame 1073/1074, at time 44.71/44.72 sec. Using the last valid frame instead.
  warnings.warn("Warning: in file %s, "%(self.filename)+



Moviepy - Done !
Moviepy - video ready final_dubbed_video.mp4
🎬✅ Final video saved to final_dubbed_video.mp4


## 🎬 Step 4: View the Final Result

In [ ]:
if os.path.exists(final_video_path):
  display(Video(final_video_path, embed=True, width=600))
else:
  print("Could not find the final video file. Please check for errors in the previous steps.")

This cell output is too large and can only be displayed while logged in.


#Here Is Dubbed Video Folder Link

Link - https://drive.google.com/drive/folders/1KA7jXcIEg9Zjl2UzGOSXXxx7aB1RG87i?usp=sharing
